# TP2 — Offline Stage  
## Full-Order FEM Solutions and POD-Based Reduced Order Model

This notebook implements the **offline stage** of **Test Problem 2 (TP2)**.  
The objective of this stage is to construct a **Reduced Order Model (ROM)** based on **Proper Orthogonal Decomposition (POD)** starting from **high-fidelity Finite Element Method (FEM) solutions**.

The offline stage corresponds to the **Offline Module of the Inference Engine** described in the paper and provides all the reduced quantities required for efficient online simulations.

In [ ]:
import sys
from pyprojroot import here

PROJECT_ROOT = str(here()) + "/"
sys.path.append(PROJECT_ROOT)
sys.dont_write_bytecode = True

In [ ]:
from paths import INV_PATH, TP2_PATH

ABS_PATH = PROJECT_ROOT + INV_PATH + TP2_PATH
LOAD_MSH = PROJECT_ROOT

model_name = "column"
xdmf_file_name = "column.xdmf"

fig_dir = "figures/"
eig_fig = ABS_PATH + fig_dir + "sv.png"
error_fig = ABS_PATH + fig_dir + "error.png"
speed_up_fig = ABS_PATH + fig_dir + "speed_up.png"
samples_fig = ABS_PATH + fig_dir + "samples.png"

dbdir = ABS_PATH + "trainPOD/"

Import of packages

In [ ]:
import fenics as fe

from fem_problems.invTP2.finite_element import SystemParabolicFEM
from fem_problems.invTP2.rbnics_pod_temp import TemporalPOD

## Parametrized System of Parabolic PDEs

We consider the parametrized system of parabolic PDEs defined on the 3D column domain $ \Omega \subset \mathbb{R}^3 $ introduced in the notebook *00_ProblemSettings.ipynb* and where the parameter vector is defined as

\begin{equation}
\mu = (\lambda, \alpha, \beta).
\tag{1}
\end{equation}

The availability of a manufactured analytical solution enables quantitative validation of the reduced-order approximation.

Load of the mesh elaborated in the notebook *00_ProblemSettings.ipynb*

In [ ]:
# Path to the XDMF file
file_path = ABS_PATH + xdmf_file_name

# Load mesh from file XDMF
mesh = fe.Mesh()
with fe.XDMFFile(file_path) as infile:
    infile.read(mesh)

## Parameters range

Here we fix the range of parameters. Specifically, $\mu = \left( \lambda, \alpha, \beta \right) \in \left[0, 1 \right]^3$

In [ ]:
mu_range = [
    (0, 1.),
    (0, 1.),
    (0, 1.),
]

In [ ]:
temporal_steps = 21
time_interval = [0, 1]

fem_p = SystemParabolicFEM(mesh, time_interval, temporal_steps)
pod = TemporalPOD(mu_range, fem_p)

## Snapshot Generation (Time-Dependent Case)

To construct the reduced-order model for the **time-dependent parabolic system**, we compute a set of **high-fidelity space–time snapshots** by solving the FEM problem for different values of the parameter vector \( \boldsymbol{\mu} \).

Let

\begin{equation}
\mu_1, \mu_2, \dots, \mu_M
\subset \mathcal{P}
\tag{2}
\end{equation}

be a set of sampled parameters, with  
$ \mu = (\lambda, \alpha, \beta) $.

Since the problem explicitly depends on time, the solution is computed over a discrete temporal grid

\begin{equation}
0 = t_0 < t_1 < \dots < t_N = T = 1 .
\tag{3}
\end{equation}

For each parameter sample $ \mu_i $ and each time step $ t_n $, a FEM solution
$ \mathbf{u}_h(\mu_i, t_n) \in \mathbb{R}^2 $
is computed and stored.

The resulting snapshot matrix collects all space–time solutions and is defined as

\begin{equation}
S =
\begin{bmatrix}
u_h(\mu_1, t_0) &
\dots &
u_h(\mu_1, t_N) &
\dots &
u_h(\mu_M, t_N)
\end{bmatrix}.
\tag{4}
\end{equation}

This formulation allows capturing both **parametric** and **temporal** variability of the solution manifold.


### Weak Formulation and FEM Discretization

Let $ V $ be a suitable Sobolev space.  
The semi-discrete weak formulation of the parabolic problem reads: find  
$ u(t) \in V $ such that

\begin{equation}
m(\dot{u}(t), v) + a(u(t), v)
= L(v; t, \mu)
\quad \forall v \in V.
\tag{5}
\end{equation}

The bilinear and linear forms are defined as

\begin{align}
m(\dot{u}, v) &= \int_{\Omega} \dot{u} \cdot v \, dx, \tag{6} \\ 
a(u, v) &= \int_{\Omega} \nabla u \cdot \nabla v \, dx, \tag{7} \\
L(v; t, \mu) &= \int_{\Omega} F(t;\mu) \cdot v \, dx .
\tag{8}
\end{align}

The problem is discretized in space using the **Finite Element Method** with first-order Lagrange elements, and in time by means of an **implicit Euler scheme**.  
The sequence of solutions obtained at each time step constitutes the snapshot set used for the POD-based reduced-order model.


In [ ]:
num_training = 20
num_testing = 5

N_modes = num_training*fem_p.temp_points
tol = 1e-6

In [ ]:
training = pod.sampling_parameters(num_training)
testing = pod.sampling_parameters(num_testing, test=True)
pod.plot_samples(figsize = (6, 6), filename=samples_fig)

## Proper Orthogonal Decomposition (POD)

The reduced basis is constructed by applying **Proper Orthogonal Decomposition (POD)** to the snapshot matrix $ \mathbf{S} $.

The POD basis $ \{ \xi_1, \dots, \xi_k \} $ is obtained by solving the eigenvalue problem associated with the correlation matrix, retaining the modes that maximize the captured energy.

The reduced space is defined as

\begin{equation}
V_{\text{rb}} = \text{span}\{\xi_1, \dots, \xi_k\}, \tag{9}
\end{equation}

where the dimension $ k $ is selected according to an energy-based tolerance criterion.

In [ ]:
pod.pod_execution(N_modes, tol)

Plot of eigenvalues

In [ ]:
pod.plot_eigenvalues(filename=eig_fig)

Storing of reduced basis

In [ ]:
pod.store_reduction(directory=dbdir, filename=model_name)

Example of resolution for specifica values of parameters with ROM, FOM, and analytical solution

In [ ]:
mu_test = [.2, .6, .4]
mat, _ = pod.solve_rom(mu_test, pod.num_basis)
fom_m, _ = pod.fem_p.solve_fem(mu_test)
real_m, _ = pod.fem_p.exact_solution(mu_test)

print("ERROR FOM-EXACT")
e, f = pod.fem_p.compute_error(fom_m, real_m)
print("ERROR ROM-FOM")
a, b = pod.fem_p.compute_error(mat, fom_m)
print("ERROR ROM-EXACT")
c, d = pod.fem_p.compute_error(mat, real_m)

## Error Analysis

To assess the accuracy of the reduced-order model, we evaluate the approximation error with respect to the full-order FEM solution.

In [ ]:
error, _ = pod.error_analysis()

Plot of error analysis results

In [ ]:
pod.plot_errors(error, filename=error_fig)

Plot of obtained speed up

In [ ]:
pod.plot_speed_up(error, filename=speed_up_fig)

Saving and showing table of obtained errors

In [ ]:
pod.save_table_error(error)

In [ ]:
pod.table_error(error)

## Offline Stage Outputs

At the end of the offline stage, the following quantities are available:

- POD basis functions and reduced operators,
- error indicators for reduced-order approximations.

These artifacts are stored and reused in the **online stage**, enabling fast and reliable simulations.